# Introduction to Strands Agents

## Overview
[Strands Agents](https://strandsagents.com) is an open-source SDK for building AI agents in a few lines of Python. The model drives the loop: it reads the conversation, decides whether to call a tool, reads the tool result, and repeats until it can answer.

This notebook builds your first agent, adds tools, calls tools directly, turns on logging and inspects what the model provider is doing. A task-specific agent, **RecipeBot**, lives in `recipe-bot-cli/` (see the week 7 README). The model provider and model are set in `config.yaml`.

The notebook uses a single agent with two custom tools (calculator, weather). Strands features covered: `Agent`, `@tool`, direct tool calls, logging and result metrics.

## What it does
* Creates an agent with a system prompt
* Adds custom tools; type hints and docstring become the tool schema
* Invokes a tool directly with `agent.tool.<name>`
* Reads the result object: stop reason, metrics, message history
* Configures logging
* Builds the model from `config.yaml`

## Setup and prerequisites

* Python 3.10+
* The model provider from `config.yaml` running and reachable, with its model available
* `pip install -r requirements.txt`

All settings come from [config.yaml](config.yaml): the model provider and model, the sampling temperature and the context window. Change them there and every notebook in this week follows.

In [1]:
#!pip install -r requirements.txt

In [2]:
import logging, warnings
import yaml
from strands import Agent, tool
from strands.models.ollama import OllamaModel

warnings.filterwarnings("ignore")
logging.getLogger("strands").setLevel(logging.WARNING)

cfg = yaml.safe_load(open("config.yaml"))

def make_model(**overrides):
    """One place to build the model from config.yaml."""
    params = dict(
        host=cfg["ollama"]["url"],
        model_id=cfg["ollama"]["models"][0],
        temperature=cfg["strands"]["temperature"],
        options={"num_ctx": cfg["strands"]["num_ctx"]},
    )
    params.update(overrides)
    return OllamaModel(**params)

model = make_model()
print("Model:", model.config["model_id"], "@", cfg["ollama"]["url"])

Model: gemma4:e2b-mlx @ http://localhost:11434


In [3]:
# Fail early and clearly if the model server is not up or the model is missing
import requests
tags = requests.get(cfg["ollama"]["url"] + "/api/tags", timeout=5).json()["models"]
available = [m["name"] for m in tags]
assert model.config["model_id"] in available, f"model not available: {model.config['model_id']}"
print("Model server is up with", len(available), "models")

Model server is up with 15 models


## Creating your first agent

An agent is a model plus a system prompt plus (optionally) tools. The architecture of the simplest case: your Python process runs the agent loop and calls the model, which is served by the provider set in `config.yaml`.

<div style="text-align:center">
    <img src="images/simple_agent.svg" width="75%" />
</div>

In [4]:
agent = Agent(
    model=model,
    system_prompt="You are a helpful assistant that provides concise responses.",
)

response = agent("Hello! Tell me a joke.")

Why did the scarecrow win an award? Because he was outstanding in his field!

The call returns an `AgentResult`. `str(response)` is the final text; the other fields tell you how the run went.

In [5]:
print("stop_reason :", response.stop_reason)
print("text         :", str(response)[:80], "...")
print("cycles       :", response.metrics.cycle_count)
print("tokens       :", response.metrics.accumulated_usage)
print("model latency:", response.metrics.accumulated_metrics)

stop_reason : end_turn
text         : Why did the scarecrow win an award? Because he was outstanding in his field!
 ...
cycles       : 1
tokens       : {'inputTokens': 33, 'outputTokens': 196, 'totalTokens': 229}
model latency: {'latencyMs': 2165}


### Conversation history

The agent keeps the conversation in `agent.messages` so a second call has context. Each message has a `role` and a list of content blocks.

In [6]:
agent("Explain that joke in one sentence.")
for m in agent.messages:
    kinds = [list(block)[0] for block in m["content"]]
    print(f'{m["role"]:>9}: {kinds}')

The joke is a pun based on the scarecrow being excellent at his job.     user: ['text']
assistant: ['text']
     user: ['text']
assistant: ['text']


## Add tools to the agent

Tools are plain Python functions with the `@tool` decorator. The **type hints** and the **docstring** are not just documentation: Strands turns them into the tool's input schema, which is what the model reads to decide *when* and *how* to call it. The [strands-agents-tools](https://github.com/strands-agents/tools) package also ships ready-made tools, but writing your own is the skill that carries over.

<div style="text-align:center">
    <img src="images/agent_with_tools.svg" width="75%" />
</div>

In [7]:
import operator
from typing import Literal

_OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv, "**": operator.pow}

@tool
def calculator(a: float, b: float, op: Literal["+", "-", "*", "/", "**"]) -> float:
    """Apply an arithmetic operator to two numbers.

    Args:
        a: Left operand.
        b: Right operand.
        op: "+" add, "-" subtract, "*" multiply, "/" divide, "**" raise a to the power b.
    """
    return _OPS[op](a, b)

@tool
def weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city to get the weather for.
    """
    # Dummy implementation - a real tool would call a weather API
    return f"It is sunny and 72°F in {city}."

agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt=("You are a helpful assistant. Use the calculator for any arithmetic and the weather tool "
                   "for weather questions, then reply in one short sentence that repeats the tool result without adding facts."),
)

response = agent("What is the weather in Seattle today?")


Tool #1: weather
It is sunny and 72°F in Seattle.

Look at the schema the model actually received for `calculator`. It was generated from the signature and docstring above:

In [8]:
import json
print(json.dumps(agent.tool_registry.get_all_tools_config()["calculator"], indent=2))

{
  "name": "calculator",
  "description": "Apply an arithmetic operator to two numbers.",
  "inputSchema": {
    "json": {
      "properties": {
        "a": {
          "description": "Left operand.",
          "type": "number"
        },
        "b": {
          "description": "Right operand.",
          "type": "number"
        },
        "op": {
          "description": "\"+\" add, \"-\" subtract, \"*\" multiply, \"/\" divide, \"**\" raise a to the power b.",
          "enum": [
            "+",
            "-",
            "*",
            "/",
            "**"
          ],
          "type": "string"
        }
      },
      "required": [
        "a",
        "b",
        "op"
      ],
      "type": "object"
    }
  }
}


### Invoking a tool directly

Sometimes you want to call a tool yourself: to debug it, to pre-load the agent with a customer's data, or to use one tool inside another. Use `agent.tool.<name>(...)`. The result is a `ToolResult` dict with a `status` and `content`.

In [9]:
result = agent.tool.calculator(a=144, b=0.5, op="**")
print(result)

{'toolUseId': 'tooluse_calculator_958747685', 'status': 'success', 'content': [{'text': '12.0'}]}


By default a direct call is also **recorded in the message history**, so the model sees it on the next turn. Set `record_direct_tool_call=False` on the agent to turn that off.

Any other Strands provider (OpenAI-compatible endpoints through LiteLLM, llama.cpp, and so on) plugs in the same way. See the [model providers guide](https://strandsagents.com/docs/user-guide/concepts/model-providers/).

## Summary

1. An **agent** is a model, a system prompt and tools. The model decides what to call.
2. A tool's **type hints and docstring are its schema**. Write them for the model.
3. `agent.tool.<name>()` calls a tool directly, and `agent.messages` / `result.metrics` show what happened.
4. Everything here ran on the model chosen in `config.yaml`. Next: more ways to define tools, and MCP servers (7.2).
